# faultmap — Find Where Your LLM Is Failing

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabonavarroo/faultmap/blob/main/notebooks/tutorial.ipynb)
[![PyPI version](https://img.shields.io/pypi/v/faultmap.svg)](https://pypi.org/project/faultmap/)

**faultmap** automatically discovers *where* and *why* your LLM is failing — using embedding-space clustering and statistical hypothesis testing.

This notebook walks through all four usage modes:
- **Mode 1**: Bring your own scores (DeepEval, Ragas, LLM-as-judge, human labels)
- **Mode 2**: Reference-based scoring (cosine similarity to ground truth)
- **Mode 3**: Fully autonomous (semantic entropy + self-consistency — no labels needed)
- **Coverage auditing**: Find what your test suite is missing vs production traffic

Each section shows both a **mock path** (runs instantly, no API key needed) and the equivalent **real API path** in comments.

In [ ]:
# Install faultmap with the rich terminal output extra
# Skip this cell if you already have it installed
!pip install faultmap[rich] -q

## The Problem

Your eval says **85% accuracy**. Users are complaining. Where are the failures?

Aggregate metrics hide critical patterns. A model can score well overall while catastrophically failing on specific input types — legal questions, billing disputes, technical setup — that matter most to your users.

```
All prompts combined:  85% pass rate  ← looks fine

Legal questions:       22% pass rate  ← disaster
Billing questions:     91% pass rate  ← fine
Setup questions:       89% pass rate  ← fine
```

faultmap answers: **"Which input slices have a statistically significantly higher failure rate?"**

## How faultmap Works

```
Prompts
  │
  ▼
Embed  ──────────────────────────────────────────────┐
  │                                                  │
  ▼                                               Coverage Audit:
Cluster (HDBSCAN or agglomerative)               compare test vs prod
  │                                                  │
  ▼                                                  ▼
Statistical Test (chi2 / Fisher exact)          Nearest-neighbor distance
  │                                                  │
  ▼                                                  ▼
BH FDR Correction                               Cluster gaps
  │                                                  │
  ▼                                                  ▼
LLM Names each slice                           LLM Names each gap
  │                                                  │
  ▼                                                  ▼
AnalysisReport                              CoverageReport
```

**No false positives at scale**: Benjamini-Hochberg FDR correction controls the false discovery rate across all clusters simultaneously.

## Setup: Mock vs Real API Keys

This tutorial uses **mocks by default** — it runs without any API key and produces deterministic results that illustrate the library perfectly.

The mock replaces:
1. The embedding model → deterministic hash-based vectors (no model download)
2. The LLM (naming clusters) → a canned response

The `SliceAnalyzer` API is **identical** — mock vs real is an internal implementation detail.

### Real API path (optional)

To use real API keys instead, set your environment variable before creating the analyzer:

```python
# Standard environment variable (works everywhere)
import os
os.environ["OPENAI_API_KEY"] = "sk-..."

# On Google Colab, use Secrets (left sidebar → key icon):
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
```

faultmap supports 100+ providers via litellm — Anthropic, Ollama, Groq, Azure, etc.

In [ ]:
# Mock infrastructure — runs without any API key
# This cell sets up deterministic embeddings and a fake LLM for the tutorial.
# In real usage, just skip this cell and set your API key instead.

import asyncio
import hashlib
import re
from unittest.mock import AsyncMock, MagicMock, patch

import numpy as np


class MockEmbedder:
    """Deterministic hash-based embedder — no model download, no API key."""

    DIM = 64

    async def embed(self, texts: list[str]) -> np.ndarray:
        vecs = []
        for t in texts:
            h = hashlib.md5(t.encode()).digest()
            arr = np.frombuffer(h * (self.DIM // 16 + 1), dtype=np.uint8)[: self.DIM]
            vec = arr.astype(np.float32) / 127.5 - 1.0
            vec /= np.linalg.norm(vec) + 1e-8
            vecs.append(vec)
        return np.array(vecs, dtype=np.float32)


def make_mock_llm_response(name="Identified Cluster", description="Prompts in this cluster share a common theme."):
    """Return a mock litellm completion response."""
    response = MagicMock()
    response.choices = [MagicMock()]
    response.choices[0].message.content = f"Name: {name}\nDescription: {description}"
    return response


# Patch targets — these replace faultmap internals during this tutorial
EMBEDDER_PATCH = "faultmap.analyzer.get_embedder"
LLM_PATCH = "faultmap.labeling.litellm.acompletion"

mock_embedder = MockEmbedder()
mock_llm = AsyncMock(return_value=make_mock_llm_response())

print("Mock infrastructure ready. Run any cell below — no API key needed.")

## Mode 1: Bring Your Own Scores

The most common use case: you already have scores from your eval framework and want to know *which input types* are responsible for failures.

Works with scores from:
- [DeepEval](https://github.com/confident-ai/deepeval): `metric.score`
- [Ragas](https://github.com/explodinggradients/ragas): `result['answer_relevancy']`
- Human annotation: `1.0` (good) / `0.0` (bad)
- Any LLM-as-judge normalized to `[0, 1]`

**Higher = better**. Scores below `failure_threshold` (default `0.5`) count as failures.

In [ ]:
from faultmap import SliceAnalyzer

# ---------------------------------------------------------------------------
# Synthetic dataset: 90 prompts, 3 topics, 1 known failure cluster
#   - Legal/compliance → low scores (~0.2) ← model struggles here
#   - Password/account reset → high scores (~0.85)
#   - Pricing questions → high scores (~0.90)
# ---------------------------------------------------------------------------

legal_topics = ["GDPR", "HIPAA", "SOC2", "PCI-DSS", "CCPA", "ISO 27001"]
account_topics = ["password", "2FA", "email", "phone", "settings", "account"]
pricing_topics = ["Basic", "Pro", "Enterprise", "Team", "Starter", "Business"]

prompts = (
    [f"What are the legal requirements for {t} compliance?" for t in legal_topics * 5]
    + [f"How do I reset my {t}?" for t in account_topics * 5]
    + [f"What is included in the {t} plan?" for t in pricing_topics * 5]
)
responses = [f"Here is information about: {p}" for p in prompts]
scores = [0.2] * 30 + [0.85] * 30 + [0.90] * 30

print(f"Dataset: {len(prompts)} prompts, {sum(s < 0.5 for s in scores)} failures")
print(f"Baseline failure rate: {sum(s < 0.5 for s in scores) / len(scores):.0%}")

In [ ]:
# Run analysis with mock (default — no API key needed)
with patch(EMBEDDER_PATCH, return_value=mock_embedder), patch(LLM_PATCH, mock_llm):
    analyzer = SliceAnalyzer(
        model="gpt-4o-mini",           # used for naming only — mocked here
        min_slice_size=5,
        failure_threshold=0.5,
        significance_level=0.05,
    )
    report = analyzer.analyze(prompts, responses, scores=scores)

# ── Real API path (remove the `with patch(...)` wrapper above and use this) ──
# import os; os.environ["OPENAI_API_KEY"] = "sk-..."
# analyzer = SliceAnalyzer(model="gpt-4o-mini", embedding_model="text-embedding-3-small")
# report = analyzer.analyze(prompts, responses, scores=scores)

print(report)

## Inspecting Results

The `AnalysisReport` gives you structured access to every failure slice.

In [ ]:
import json

# Top-level statistics
print(f"Total prompts:        {report.total_prompts}")
print(f"Total failures:       {report.total_failures}")
print(f"Baseline failure rate: {report.baseline_failure_rate:.1%}")
print(f"Significant slices:   {report.num_significant}")
print()

# Each failure slice
for s in report.slices:
    print(f"Slice: {s.name!r}")
    print(f"  Description:    {s.description}")
    print(f"  Size:           {s.size} prompts")
    print(f"  Failure rate:   {s.failure_rate:.0%} (vs {s.baseline_rate:.0%} baseline)")
    print(f"  Effect size:    {s.effect_size:.1f}x worse than baseline")
    print(f"  p-value (adj):  {s.adjusted_p_value:.4f}")
    print(f"  Test used:      {s.test_used}")
    print(f"  Sample prompts:")
    for p in s.representative_prompts[:3]:
        print(f"    - {p}")
    print()

# Export to JSON
report_dict = report.to_dict()
print(f"JSON keys: {list(report_dict.keys())}")
# json.dumps(report_dict)  # fully serializable

## Mode 2: Reference-Based Scoring

Provide ground-truth answers. faultmap computes cosine similarity between each response and its reference embedding — no LLM judge needed.

Best for: RAG evaluation, Q&A benchmarks, translation quality.

In [ ]:
# Dataset: 60 prompts where legal answers are deliberately wrong
qa_prompts = (
    [f"What does {reg} regulate?" for reg in ["GDPR", "HIPAA", "CCPA", "SOC2", "PCI-DSS"] * 6]
    + [f"How do I set up {feature}?" for feature in ["SSO", "2FA", "webhooks", "API keys", "SAML"] * 6]
)

# Ground truth references
references = (
    [f"{reg} is a data protection regulation governing personal data handling." for reg in ["GDPR", "HIPAA", "CCPA", "SOC2", "PCI-DSS"] * 6]
    + [f"To set up {feature}, navigate to Settings and follow the configuration steps." for feature in ["SSO", "2FA", "webhooks", "API keys", "SAML"] * 6]
)

# Responses: legal answers are hallucinated (completely off-topic)
qa_responses = (
    [f"I recommend checking the pricing page for details." for _ in range(30)]  # wrong!
    + [f"To set up {feature}, navigate to Settings and follow the configuration steps." for feature in ["SSO", "2FA", "webhooks", "API keys", "SAML"] * 6]  # correct
)

print(f"Dataset: {len(qa_prompts)} prompts")

In [ ]:
with patch(EMBEDDER_PATCH, return_value=mock_embedder), patch(LLM_PATCH, mock_llm):
    analyzer2 = SliceAnalyzer(
        model="gpt-4o-mini",
        min_slice_size=5,
        failure_threshold=0.5,
    )
    # Pass references instead of scores → Mode 2
    report2 = analyzer2.analyze(qa_prompts, qa_responses, references=references)

# ── Real API path ──
# report2 = analyzer.analyze(qa_prompts, qa_responses, references=references)

print(report2)
print(f"\nBaseline failure rate: {report2.baseline_failure_rate:.1%}")
print(f"Significant slices: {report2.num_significant}")

## Mode 3: Reference-Free / Autonomous

No ground truth needed. faultmap estimates reliability by:
1. Sampling `n_samples` additional responses per prompt at `temperature=1.0`
2. Measuring **semantic entropy**: how spread out are the responses? (high spread = uncertain)
3. Measuring **self-consistency**: what fraction of samples agree with the original?
4. Combining: `score = 0.5 × (1 − entropy) + 0.5 × consistency`

High-entropy clusters reveal where the model is uncertain or hallucinating — with **no labels required**.

Best for: discovering unknown unknowns in production traffic.

In [ ]:
# For Mode 3 we also need to mock the LLM sampling calls (litellm.acompletion)
# The mock returns diverse responses for legal prompts (high entropy)
# and consistent responses for setup prompts (low entropy)

mode3_prompts = (
    [f"Explain {reg} compliance requirements in detail." for reg in ["GDPR", "HIPAA", "CCPA", "SOC2"] * 5]
    + [f"How do I configure {f} for my team?" for f in ["SSO", "2FA", "webhooks", "LDAP"] * 5]
)
mode3_responses = [f"Response to: {p}" for p in mode3_prompts]

call_count = [0]

async def diverse_llm_mock(*args, **kwargs):
    """Returns varied responses for legal prompts, consistent ones for setup prompts."""
    messages = kwargs.get("messages", [])
    prompt_text = messages[-1]["content"] if messages else ""
    call_count[0] += 1

    legal_keywords = ["GDPR", "HIPAA", "CCPA", "SOC2", "compliance"]
    is_legal = any(kw.lower() in prompt_text.lower() for kw in legal_keywords)

    # Naming calls return Name/Description format
    if "Name:" in prompt_text or "name" in prompt_text.lower():
        resp = make_mock_llm_response("Legal Compliance Questions", "Prompts about regulatory compliance.")
        return resp

    # Sampling calls: legal = diverse (high entropy), setup = consistent
    diverse_answers = [
        "This regulation primarily concerns data residency requirements.",
        "You need to appoint a Data Protection Officer and conduct impact assessments.",
        "The key requirement is obtaining explicit consent before processing personal data.",
        "Organizations must report breaches within 72 hours to the supervisory authority.",
        "Compliance requires annual third-party audits and penetration testing.",
        "The regulation mandates data minimization and purpose limitation principles.",
        "You must implement technical and organizational measures for data security.",
        "Cross-border data transfers require Standard Contractual Clauses or adequacy decisions.",
    ]
    consistent_answer = "Navigate to Settings > Security and follow the step-by-step wizard."

    resp = MagicMock()
    resp.choices = [MagicMock()]
    resp.choices[0].message.content = diverse_answers[call_count[0] % len(diverse_answers)] if is_legal else consistent_answer
    return resp

print(f"Mode 3 dataset: {len(mode3_prompts)} prompts")

In [ ]:
# Mode 3: no scores, no references — faultmap scores autonomously
with patch(EMBEDDER_PATCH, return_value=mock_embedder), patch(LLM_PATCH, diverse_llm_mock):
    analyzer3 = SliceAnalyzer(
        model="gpt-4o-mini",
        min_slice_size=5,
        n_samples=4,           # responses sampled per prompt for entropy estimation
        temperature=1.0,       # high temperature = diverse samples = better entropy signal
    )
    report3 = analyzer3.analyze(mode3_prompts, mode3_responses)  # no scores, no references

# ── Real API path ──
# report3 = analyzer.analyze(mode3_prompts, mode3_responses)  # just omit scores + references

print(report3)
print(f"\nBaseline failure rate: {report3.baseline_failure_rate:.1%}")
print(f"Significant slices: {report3.num_significant}")

## Coverage Auditing

Find out what your test suite is **missing** by comparing it against production traffic.

faultmap finds production prompts that have no semantically nearby test prompt — then clusters them into coherent gap descriptions.

Use this to:
- Prioritize which new test cases to write
- Discover unknown user intents not covered by your test suite
- Track coverage improvement over time

In [ ]:
# Test suite: covers billing and account topics
test_prompts = (
    [f"How do I {action} my account?" for action in ["upgrade", "downgrade", "cancel", "pause", "reactivate"] * 4]
    + [f"What does the {plan} plan include?" for plan in ["Basic", "Pro", "Enterprise", "Team"] * 5]
)

# Production traffic: covers the above PLUS a new topic the test suite never touched
production_prompts = (
    # Covered topics (similar to test set)
    [f"How do I {action} my subscription?" for action in ["upgrade", "downgrade", "cancel", "pause", "modify"] * 4]
    + [f"Tell me about the {plan} tier pricing." for plan in ["Basic", "Pro", "Enterprise", "Team"] * 4]
    # Gap: API integration questions — never seen in test suite!
    + [f"How do I integrate faultmap with {tool}?" for tool in ["GitHub Actions", "LangChain", "Pytest", "DeepEval", "Weights & Biases", "MLflow"] * 3]
    + [f"Can I use faultmap with {provider}?" for provider in ["AWS Bedrock", "Azure OpenAI", "Hugging Face", "Ollama"] * 3]
)

print(f"Test prompts:        {len(test_prompts)}")
print(f"Production prompts:  {len(production_prompts)}")
print(f"Known gap size:      {6*3 + 4*3} prompts (API integration)")

In [ ]:
with patch(EMBEDDER_PATCH, return_value=mock_embedder), patch(LLM_PATCH, mock_llm):
    coverage = analyzer.audit_coverage(
        test_prompts=test_prompts,
        production_prompts=production_prompts,
        distance_threshold=1.0,  # explicit threshold; None = auto (mean + 1.5*std)
        min_gap_size=3,
    )

# ── Real API path ──
# coverage = analyzer.audit_coverage(test_prompts, production_prompts)

print(coverage)

## Interpreting & Exporting Results

In [ ]:
# Coverage report
print(f"Overall coverage score: {coverage.overall_coverage_score:.0%}")
print(f"Coverage gaps found:    {coverage.num_gaps}")
print()

for gap in coverage.gaps:
    print(f"Gap: {gap.name!r}")
    print(f"  Description:        {gap.description}")
    print(f"  Size:               {gap.size} uncovered production prompts")
    print(f"  Mean distance:      {gap.mean_distance:.2f} (higher = further from any test prompt)")
    print(f"  Sample prompts:")
    for p in gap.representative_prompts[:3]:
        print(f"    - {p}")
    print()

# Prioritize failure slices by effect size
if report.slices:
    print("\nFailure slices ranked by effect size (worst first):")
    for s in sorted(report.slices, key=lambda x: x.effect_size, reverse=True):
        print(f"  {s.effect_size:.1f}x worse — {s.name!r} ({s.size} prompts, {s.failure_rate:.0%} failure rate)")

# Export everything to JSON
import json
analysis_json = json.dumps(report.to_dict(), indent=2)
coverage_json = json.dumps(coverage.to_dict(), indent=2)
print(f"\nAnalysis report: {len(analysis_json)} chars of JSON")
print(f"Coverage report: {len(coverage_json)} chars of JSON")

## Real Usage Quick Start

Here's the complete code for all four modes with a real API key — copy and adapt:

In [ ]:
# Complete real-usage code — requires OPENAI_API_KEY (or any litellm provider)
# Uncomment and run after setting your API key

"""
import os
from faultmap import SliceAnalyzer

# Set API key
os.environ["OPENAI_API_KEY"] = "sk-..."  # or use Colab secrets

# Create analyzer — one instance for all modes
analyzer = SliceAnalyzer(
    model="gpt-4o-mini",                       # LLM for naming slices
    embedding_model="text-embedding-3-small",  # or "all-MiniLM-L6-v2" for local
    significance_level=0.05,
    min_slice_size=10,
    failure_threshold=0.5,
)

# Mode 1: use scores from your eval framework
report = analyzer.analyze(prompts, responses, scores=your_scores)

# Mode 2: use ground-truth references
report = analyzer.analyze(prompts, responses, references=ground_truth_answers)

# Mode 3: no labels — fully autonomous
report = analyzer.analyze(prompts, responses)

# Coverage audit
coverage = analyzer.audit_coverage(
    test_prompts=my_test_suite,
    production_prompts=my_production_logs,
)

# Display and export
print(report)
print(coverage)
import json
print(json.dumps(report.to_dict(), indent=2))
"""

print("Uncomment the code above to run with real API keys.")

## Next Steps

- **[README](https://github.com/gabonavarroo/faultmap#readme)** — full API reference, scoring mode details, embedding options, statistical methodology
- **[Examples](https://github.com/gabonavarroo/faultmap/tree/main/examples)** — runnable scripts for each mode
- **[Issues](https://github.com/gabonavarroo/faultmap/issues)** — bug reports and feature requests

### Key parameters to tune

| Parameter | Default | When to change |
|-----------|---------|----------------|
| `min_slice_size` | `10` | Lower for small datasets (<200 prompts) |
| `failure_threshold` | `0.5` | Match your eval framework's passing threshold |
| `significance_level` | `0.05` | Raise to `0.10` to catch more (with more false positives) |
| `n_samples` | `8` | Mode 3 only: higher = more accurate entropy, more API calls |
| `clustering_method` | `"hdbscan"` | Try `"agglomerative"` if HDBSCAN produces too few clusters |
| `embedding_model` | `"text-embedding-3-small"` | Use `"all-MiniLM-L6-v2"` for local (no API cost) |

### Supported LLM providers

faultmap uses [litellm](https://github.com/BerriAI/litellm) — any of its 100+ providers work:

```python
SliceAnalyzer(model="anthropic/claude-3-haiku-20240307")  # Anthropic
SliceAnalyzer(model="ollama/mistral")                     # Local via Ollama
SliceAnalyzer(model="groq/llama3-8b-8192")               # Groq (fast + free tier)
SliceAnalyzer(model="azure/gpt-4o-mini")                  # Azure OpenAI
```